<a href="https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_07/03_chronos_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Zeitreihenvorhersage mit Chronos-2

Dieses Notebook demonstriert die Verwendung des **Chronos-2** Modells von Amazon Science für die Vorhersage von Zeitreihen. Wir verwenden denselben Datensatz wie in `01_sequences_rnns.ipynb` (Chicago Ridership).

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_07/03_chronos_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_07/03_chronos_forecasting.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

## Setup

In [ ]:
!pip install chronos-forecasting

In [ ]:
import sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from chronos import Chronos2Pipeline

assert sys.version_info >= (3, 7)

In [ ]:
plt.rcParams.update({'font.size': 16})
plt.rc('axes', labelsize=16, titlesize=16)
plt.rc('legend', fontsize=16)
plt.rc('xtick', labelsize=12)
plt.rc('ytick', labelsize=12)

## Daten laden

In [ ]:
import tensorflow as tf

filepath = tf.keras.utils.get_file(
    "ridership.tgz",
    "https://github.com/ageron/data/raw/main/ridership.tgz",
    cache_dir=".",
    extract=True
)
if "_extracted" in filepath:
    ridership_path = Path(filepath) / "ridership"
else:
    ridership_path = Path(filepath).with_name("ridership")

path = ridership_path / "CTA_-_Ridership_-_Daily_Boarding_Totals.csv"
df = pd.read_csv(path, parse_dates=["service_date"])
df.columns = ["date", "day_type", "bus", "rail", "total"]  # shorter names
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)  # no need for total, it's just bus + rail
df = df.drop_duplicates()  # remove duplicated months (2011-10 and 2014-07)

In [ ]:
rail_train = df["rail"]["2016-01":"2018-12"] / 1e6
rail_valid = df["rail"]["2019-01":"2019-05"] / 1e6
rail_test = df["rail"]["2019-06":] / 1e6

## Chronos-2 Modell laden

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",  # 'tiny' is faster for demonstration, 'base' or 'large' are alternatives
    device_map=device
)

## Vorhersage

In [ ]:
context = torch.tensor(rail_valid.values).unsqueeze(0).unsqueeze(0)
prediction_length = len(rail_test)

forecast = pipeline.predict(
    context,
    prediction_length=prediction_length
    # num_samples=20
)

## Visualisierung

In [ ]:
low, median, high = np.quantile(forecast[0].numpy(), [0.1, 0.5, 0.9], axis=0)

print(low[0], high[0])

plt.figure(figsize=(12, 6))
plt.plot(rail_valid.index[-100:], rail_valid.values[-100:], label="Historische Daten")
plt.plot(rail_test.index, rail_test.values, label="Tatsächliche Daten")
plt.plot(rail_test.index, median[0], label="Chronos Median Vorhersage")
plt.fill_between(rail_test.index, low[0], high[0], color="red", alpha=0.3, label="80% Konfidenzintervall")
plt.legend()
plt.grid(True)
plt.xlabel("Datum")
plt.ylabel("Fahrgastzahlen (Millionen)")
plt.title("Chronos-2 Vorhersage für Chicago Rail Ridership")
plt.show()